# Pertemuan 12 : Asosiasi Data & Sistem Rekomendasi Dasar
### 

| | |
|---|---|
| **Nama Lengkap** | Nisa Agustina Maesaroh |
| **NIM** | 240401070509 |
| **Kelas** | IF403 |

In [1]:
#Langkah 1: Generate dan Eksplorasi Dataset
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur',
        'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']

#buat transaksi, tiap transaksi berisi 2-5 produk
transaksi = []
for _ in range(50):
    n_item = np.random.randint(2, 6)
    transaksi.append(list(np.random.choice(produk, n_item, replace=False)))

#suntikan pola: roit sering bersama selai
for i in range(0, 20):
    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
        transaksi[i].append('Selai')

#print(f'Transaksi:\n{transaksi}')
print('Contoh Transaksi:', transaksi[:3])
print('\nJumlah Transaksi:', len(transaksi))

Contoh Transaksi: [[np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai'], [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')], [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]]

Jumlah Transaksi: 50


In [2]:
#Langkah 2: One-Hot Encoding Transaksi
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)

df = pd.DataFrame(te_ary, columns=te.columns_)
#df.columns = df.columns.astype(str) 

print(f"Shape df: {df.shape}")
print("Kolom:", df.columns.tolist())
print(df.head())

Shape df: (50, 10)
Kolom: [np.str_('Gula'), np.str_('Keju'), np.str_('Kopi'), np.str_('Mentega'), np.str_('Roti'), 'Selai', np.str_('Sereal'), np.str_('Susu'), np.str_('Teh'), np.str_('Telur')]
    Gula   Keju   Kopi  Mentega   Roti  Selai  Sereal   Susu    Teh  Telur
0  False   True   True     True   True   True   False  False  False  False
1  False  False   True     True   True   True   False  False   True  False
2  False  False   True    False  False  False   False   True   True  False
3  False   True  False    False  False   True   False  False   True   True
4   True   True  False     True  False  False   False   True  False  False


In [3]:
#Langkah 3: Cari Frequent Itemset dengan Apriori
from mlxtend.frequent_patterns import apriori

for ms in [0.05, 0.1, 0.2]:
    freq = apriori(df, min_support=ms, use_colnames=True)
    print(f'min_support={ms}: {len(freq)} itemset ditemukan')

# Gunakan min_support yang menghasilkan jumlah itemset wajar (tidak 0, tidak ratusan)
freq_items = apriori(df, min_support=0.1, use_colnames=True)
freq_items = freq_items.sort_values('support', ascending=False)
print(f"\nItemset yang ditemukan ({len(freq_items)} itemset):")
print(freq_items.head(10))

min_support=0.05: 74 itemset ditemukan
min_support=0.1: 44 itemset ditemukan
min_support=0.2: 13 itemset ditemukan

Itemset yang ditemukan (44 itemset):
    support                 itemsets
5      0.52       frozenset({Selai})
8      0.46         frozenset({Teh})
3      0.42     frozenset({Mentega})
9      0.36       frozenset({Telur})
1      0.34        frozenset({Keju})
0      0.32        frozenset({Gula})
2      0.32        frozenset({Kopi})
4      0.32        frozenset({Roti})
7      0.32        frozenset({Susu})
36     0.24  frozenset({Selai, Teh})


In [4]:
#Langkah 4: Bentuk & Saring Aturan Asosiasi
from mlxtend.frequent_patterns import association_rules

def clean_itemset(itemset):
    return frozenset([str(item) for item in itemset])

freq_items['itemsets'] = freq_items['itemsets'].apply(clean_itemset)

if len(freq_items) > 0:
    rules = association_rules(freq_items, metric='confidence', min_threshold=0.5)
    rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)
    print("\n=== Aturan Asosiasi ===")
    print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))
else:
    print("Tidak ada itemset yang ditemukan. Coba turunkan min_support.")


=== Aturan Asosiasi ===
                    antecedents           consequents  support  confidence  \
10       frozenset({Keju, Teh})    frozenset({Telur})     0.12    0.857143   
15  frozenset({Mentega, Selai})     frozenset({Kopi})     0.10    0.625000   
11      frozenset({Gula, Roti})    frozenset({Selai})     0.10    1.000000   
7           frozenset({Sereal})  frozenset({Mentega})     0.14    0.777778   
9       frozenset({Telur, Teh})     frozenset({Keju})     0.12    0.600000   
14     frozenset({Kopi, Selai})  frozenset({Mentega})     0.10    0.714286   
8      frozenset({Keju, Telur})      frozenset({Teh})     0.12    0.750000   
12     frozenset({Gula, Selai})     frozenset({Roti})     0.10    0.500000   
13   frozenset({Kopi, Mentega})    frozenset({Selai})     0.10    0.714286   
1             frozenset({Roti})    frozenset({Selai})     0.22    0.687500   

        lift  
10  2.380952  
15  1.953125  
11  1.923077  
7   1.851852  
9   1.764706  
14  1.700680  
8   1.63043

### Interpretasi Aturan Asosiasi

#### Aturan Paling Kuat (Lift Tertinggi)
Dari hasil output, aturan dengan lift tertinggi adalah:
- **{Keju, Teh} → {Telur}** dengan lift = **2.38**

Ini berarti pelanggan yang membeli Keju dan Teh cenderung juga membeli Telur, dan hubungan ini **2.38 kali** lebih mungkin terjadi dibandingkan jika pembelian terjadi secara acak.

#### Aturan Penting Lainnya:
| Aturan | Lift | Confidence | Support |
|--------|------|------------|---------|
| {Keju, Teh} → {Telur} | 2.38 | 85.7% | 12% |
| {Mentega, Selai} → {Kopi} | 1.95 | 62.5% | 10% |
| {Gula, Roti} → {Selai} | 1.92 | **100%** | 10% |
| {Sereal} → {Mentega} | 1.85 | 77.8% | 14% |
| {Telur, Teh} → {Keju} | 1.76 | 60% | 12% |

#### Analisis Bisnis
**Apakah masuk akal secara bisnis?**

✅ **Sangat masuk akal!** 

1. **{Keju, Teh} → {Telur}**: Kombinasi ini masuk akal untuk pembelian **bahan sarapan** (omelet dengan keju + teh hangat).

2. **{Mentega, Selai} → {Kopi}**: Pelanggan yang membeli pelengkap roti (mentega & selai) cenderung membeli kopi untuk sarapan.

3. **{Gula, Roti} → {Selai}**: **Confidence 100%** - setiap pelanggan yang membeli Gula dan Roti pasti membeli Selai! Kombinasi klasik membuat **roti selai** dengan gula sebagai pemanis.

4. **{Roti} → {Selai}** (Lift 1.32, Support 22%): Ini adalah **pola paling umum** - hampir 1 dari 4 transaksi membeli kedua produk ini.

**Implikasi Bisnis:**
- **Strategi Cross-Selling**: 
  - Letakkan **Keju, Teh, dan Telur** berdekatan di rak (promosi "Bahan Omelet Sarapan")
  - Tempatkan **Mentega, Selai, dan Kopi** dalam satu area "Sarapan"

- **Promosi Bundle**: 
  - Bundle "Sarapan Sehat": Roti + Selai + Teh + Telur
  - Bundle "Kopi Pagi": Kopi + Gula + Roti + Selai

- **Rekomendasi Online**: 
  - "Pembeli Keju & Teh juga membeli Telur"
  - "Pembeli Roti & Gula juga membeli Selai"

- **Stok Barang**: Pastikan produk-produk komplementer selalu tersedia bersama.

#### Metrik yang Perhatikan:
- **Support**: Seberapa sering pola ini terjadi 
  - Tertinggi: **Roti → Selai (22%)** → pola paling umum
- **Confidence**: Keandalan aturan 
  - Tertinggi: **Gula+Roti → Selai (100%)** → selalu terjadi!
- **Lift**: Kekuatan hubungan (lift > 1 = hubungan positif)
  - Tertinggi: **Keju+Teh → Telur (2.38)** → hubungan paling kuat

In [5]:
#Langkah 5: Rekomender Sederhana dengan Content-Based Filtering
from sklearn.metrics.pairwise import cosine_similarity

katalog = pd.DataFrame({
    'produk': produk,
    'kategori': ['Bakery','Bakery','Dairy','Bakery','Dairy',
            'Dairy','Minuman','Bumbu','Minuman','Dairy']
})

fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)

def rekomendasi_serupa(nama_produk, top_n=3):
    idx = katalog.index[katalog['produk'] == nama_produk][0]
    skor = list(enumerate(sim_matrix[idx]))
    skor = sorted(skor, key=lambda x: x[1], reverse=True)
    skor = [s for s in skor if s[0] != idx][:top_n]
    return katalog.iloc[[i for i, _ in skor]]['produk'].tolist()
    
print('Mirip dengan Roti:', rekomendasi_serupa('Roti'))

Mirip dengan Roti: ['Selai', 'Sereal', 'Susu']


In [6]:
#Langkah 6: Bandingkan Kedua Pendekatan
produk_target = 'Roti'

# Dari association rules: cari consequents dari aturan yang antecedent-nya mengandung produk_target
rules_terkait = rules[rules['antecedents'].apply(
    lambda x: produk_target in x)]
    
print('Rekomendasi dari Association Rules:')
print(rules_terkait[['consequents', 'lift']].head())
print('Rekomendasi dari Content-Based:', rekomendasi_serupa(produk_target))
# Diskusikan: apakah kedua pendekatan memberi rekomendasi yang konsisten?
# Kapan sebaiknya menggunakan salah satu, atau menggabungkan keduanya (hybrid)?

Rekomendasi dari Association Rules:
           consequents      lift
11  frozenset({Selai})  1.923077
1   frozenset({Selai})  1.322115
Rekomendasi dari Content-Based: ['Selai', 'Sereal', 'Susu']


**Konsistensi:** Kedua metode sepakat merekomendasikan **Selai** untuk pembeli Roti.

**Perbedaan:** Association Rules hanya memberi Selai, Content-Based memberi Selai + Sereal + Susu (produk sekategori).

**Kapan Pakai:**
- Association Rules: saat ada data transaksi cukup dan ingin tahu pola pembelian
- Content-Based: saat produk baru atau data transaksi sedikit
- Hybrid: untuk hasil terbaik - gabungkan keduanya!

**Rekomendasi Hybrid untuk Roti:** 
1. Selai (dari kedua metode)
2. Sereal (dari content-based)
3. Susu (dari content-based)
4. Mentega (dari aturan lain)
5. Telur (dari aturan lain)

# Kesimpulan

## Apa yang Dipelajari
Dari praktikum ini, saya mempelajari:
1. **Association Rule Mining**: Menggunakan algoritma Apriori untuk menemukan pola hubungan antar produk yang sering dibeli bersamaan (misal: Roti → Selai).
2. **One-Hot Encoding**: Mengubah data transaksi menjadi format biner (0/1) menggunakan TransactionEncoder agar bisa diproses oleh algoritma Apriori.
3. **Content-Based Filtering**: Membuat sistem rekomendasi berdasarkan kemiripan kategori produk menggunakan cosine similarity.
4. **Evaluasi Rekomendasi**: Membandingkan pendekatan collaborative (association rules) dengan content-based filtering.

## Temuan Utama
- **Association Rules** menemukan bahwa produk Roti dan Selai memiliki hubungan kuat (lift tinggi), yang masuk akal secara bisnis karena sering dikonsumsi bersamaan.
- **Content-Based Filtering** merekomendasikan produk berdasarkan kemiripan kategori (misal: Roti → Sereal karena sama-sama Bakery).
- Kedua pendekatan memberikan rekomendasi yang berbeda: association rules berdasarkan pola pembelian, sedangkan content-based berdasarkan atribut produk.
- Kombinasi kedua metode (hybrid) dapat memberikan rekomendasi yang lebih komprehensif.

## Keterbatasan & Pertanyaan
- **Keterbatasan**: 
  - Data transaksi hanya 50 sampel, kurang representatif untuk analisis yang akurat.
  - Content-Based hanya menggunakan 1 fitur (kategori), belum mempertimbangkan fitur lain seperti harga atau brand.
  - Association Rules hanya mempertimbangkan 2-5 item per transaksi.
  
- **Pertanyaan**:
  1. Bagaimana performa algoritma Apriori jika dataset diperbesar menjadi ribuan transaksi?
  2. Apakah ada metode lain yang lebih baik untuk sistem rekomendasi e-commerce?
  3. Bagaimana cara menggabungkan association rules dan content-based filtering menjadi hybrid recommender?
